# 06 · Model Serving

## 이 노트북에서

1. **Endpoint 생성** — Databricks SDK + REST
2. **호출 4가지 방법** — `mlflow.deployments`, `WorkspaceClient.serving_endpoints.query()`, `requests`, `curl`
3. **AI Gateway Inference Tables** — 요청/응답 자동 로깅 (legacy `auto_capture_config` 대체)
4. **Blue/Green canary** — `traffic_config` 로 점진적 롤아웃
5. **Batch scoring** 가이드 — Spark UDF 가 정답

In [ ]:
%pip install -q "databricks-sdk>=0.30.0" "mlflow>=2.20.0"
%restart_python

In [ ]:
%run ./config

In [ ]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

## Step 1. 배포할 모델 버전 확인

`05_uv_wheel` 에서 등록한 모델 사용. 없으면 그 노트북 먼저 실행하세요.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
versions = client.search_model_versions(f"name='{model_wheel}'")
versions_sorted = sorted(versions, key=lambda v: int(v.version), reverse=True)
latest = versions_sorted[0]
print(f"latest: {model_wheel} v{latest.version}")
for v in versions_sorted[:3]:
    aliases = v.aliases or []
    print(f"  v{v.version}  aliases={aliases}")

## Step 2. Endpoint 생성 — Databricks SDK

### 주요 파라미터

| 파라미터 | 의미 |
| --- | --- |
| `workload_size` | `Small` (0-4 concurrency), `Medium` (8-16), `Large` (16-64) |
| `workload_type` | `CPU` (default), `CPU_MEDIUM`, `GPU_SMALL` (T4), `GPU_LARGE` (A10G) |
| `scale_to_zero_enabled` | idle 시 $0, cold start trade-off |
| `environment_vars` | `{{secrets/scope/key}}` 형식으로 secret 참조 가능 |
| `ai_gateway` | inference table / rate limit / usage tracking — 2026 권장 |

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput, TrafficConfig, Route,
    AiGatewayConfig, AiGatewayInferenceTableConfig, AiGatewayRateLimit,
)

w = WorkspaceClient()

# 첫 served entity 이름 (logical alias, traffic_config 에서 참조)
served_name_v1 = f"wheel-v{latest.version}"

endpoint = w.serving_endpoints.create_and_wait(
    name=endpoint_wheel,
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                name=served_name_v1,
                entity_name=model_wheel,
                entity_version=str(latest.version),
                workload_size="Small",
                workload_type="CPU",
                scale_to_zero_enabled=True,   # 데모 / 개발: True. 프로덕션: False
                environment_vars={
                    "LOG_LEVEL": "INFO",
                    # "AZURE_OPENAI_KEY": "{{secrets/samsung_scope/aoai_key}}",
                },
            )
        ],
        traffic_config=TrafficConfig(
            routes=[Route(served_model_name=served_name_v1, traffic_percentage=100)],
        ),
    ),
    ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
            enabled=True,
            catalog_name=catalog,
            schema_name=schema,
            table_name_prefix=f"{endpoint_wheel.replace('-', '_')}_inference",
        ),
        rate_limits=[
            AiGatewayRateLimit(calls=300, key="endpoint", renewal_period="minute"),
        ],
    ),
    tags=[{"key": "demo", "value": "samsung-hands-on"}],
)

print(f"✓ endpoint {endpoint.name} state={endpoint.state.ready}")
print(f"   URL: https://{w.config.host}/serving-endpoints/{endpoint.name}/invocations")

## Step 3. 호출 방법 4가지

### 입력 포맷 선택

| 포맷 | 용도 |
| --- | --- |
| `dataframe_split` | **권장** — 컬럼 순서 보장 |
| `dataframe_records` | row 단위 JSON, 작은 payload 에 편함 |
| `instances` / `inputs` | tensor 모델 |

In [ ]:
import pandas as pd

pdf = spark.table(f"{catalog}.{schema}.customers").toPandas()
sample = pdf[["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets"]].iloc[:3]
display(sample)

### 방법 A — `mlflow.deployments` (노트북에서 가장 간단)

In [ ]:
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client("databricks")
resp = deploy_client.predict(
    endpoint=endpoint_wheel,
    inputs={"dataframe_split": {
        "columns": sample.columns.tolist(),
        "data":    sample.values.tolist(),
    }},
)
print(resp)

### 방법 B — Databricks SDK `serving_endpoints.query()`

In [ ]:
from databricks.sdk.service.serving import DataframeSplitInput

resp = w.serving_endpoints.query(
    name=endpoint_wheel,
    dataframe_split=DataframeSplitInput(
        columns=sample.columns.tolist(),
        data=sample.values.tolist(),
    ),
)
print(resp.predictions)

### 방법 C — `requests` (외부 앱 / 마이크로서비스 통합)

In [ ]:
import os, json, requests

token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)
host = w.config.host
url = f"{host}/serving-endpoints/{endpoint_wheel}/invocations"

resp = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"dataframe_records": sample.to_dict(orient="records")},
    timeout=30,
)
print(resp.status_code, resp.json())

### 방법 D — `curl` (CI / 운영 디버깅)

In [ ]:
import os
os.environ["DATABRICKS_HOST"] = host
os.environ["DATABRICKS_TOKEN"] = token
os.environ["ENDPOINT_NAME"] = endpoint_wheel

In [ ]:
%sh
curl -s -X POST \
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \
  -H "Content-Type: application/json" \
  -d '{"dataframe_split":{"columns":["age","tenure_months","monthly_charges","total_charges","support_tickets"],"data":[[42,18,89.5,1611.0,5]]}}' \
  "$DATABRICKS_HOST/serving-endpoints/$ENDPOINT_NAME/invocations" | python -m json.tool

## Step 4. Inference Table 확인

AI Gateway 가 요청/응답을 약 1시간 내에 Delta 테이블에 기록합니다.
데모에서는 시간이 걸리므로 **테이블이 생성되었는지만 확인**.

In [ ]:
table_prefix = f"{endpoint_wheel.replace('-', '_')}_inference"
tables = (
    spark.sql(f"SHOW TABLES IN {catalog}.{schema}")
         .filter(f"tableName LIKE '{table_prefix}%'")
)
display(tables)

## Step 5. Endpoint 메트릭

In [ ]:
ep = w.serving_endpoints.get(name=endpoint_wheel)
print(f"state.ready          = {ep.state.ready}")
print(f"state.config_update  = {ep.state.config_update}")
print(f"creator              = {ep.creator}")
print(f"creation_timestamp   = {ep.creation_timestamp}")
print(f"served_entities      :")
for se in ep.config.served_entities:
    print(f"  - {se.name}: {se.entity_name} v{se.entity_version} ({se.workload_size}/{se.workload_type})")
print(f"traffic_config       :")
for r in ep.config.traffic_config.routes:
    print(f"  - {r.served_model_name}: {r.traffic_percentage}%")

## Step 6. Blue/Green Canary 업데이트

새 모델 버전을 등록했다고 가정 (여기선 동일 모델을 served entity 만 추가).
`update_config_and_wait` 으로 두 entity 를 같이 띄우고 traffic 비율 조정.

> **실 운영 시나리오:** 새 model version (v2) 등록 → canary 10% → 메트릭 / drift 확인 → 100% → 구버전 제거

In [ ]:
# 데모: v_latest 를 두 번째 served entity로 함께 띄움 (실제로는 v_latest + 1 사용)
served_name_v2 = f"wheel-v{latest.version}-canary"

w.serving_endpoints.update_config_and_wait(
    name=endpoint_wheel,
    served_entities=[
        ServedEntityInput(
            name=served_name_v1,
            entity_name=model_wheel, entity_version=str(latest.version),
            workload_size="Small", workload_type="CPU",
            scale_to_zero_enabled=True,
        ),
        ServedEntityInput(
            name=served_name_v2,
            entity_name=model_wheel, entity_version=str(latest.version),
            workload_size="Small", workload_type="CPU",
            scale_to_zero_enabled=True,
        ),
    ],
    traffic_config=TrafficConfig(routes=[
        Route(served_model_name=served_name_v1, traffic_percentage=90),
        Route(served_model_name=served_name_v2, traffic_percentage=10),
    ]),
)
print("✓ canary 10% routed to v2")

### 100% 컷오버

In [ ]:
# 충분한 검증 후
w.serving_endpoints.update_config_and_wait(
    name=endpoint_wheel,
    served_entities=[
        ServedEntityInput(
            name=served_name_v2,
            entity_name=model_wheel, entity_version=str(latest.version),
            workload_size="Small", workload_type="CPU",
            scale_to_zero_enabled=True,
        ),
    ],
    traffic_config=TrafficConfig(routes=[
        Route(served_model_name=served_name_v2, traffic_percentage=100),
    ]),
)
print("✓ 100% on v2, v1 removed")

## Step 7. Batch Scoring — Spark UDF 가 정답

Model Serving 엔드포인트:
- **payload 16 MB cap**
- **request timeout 120s** (sync)
- 워크스페이스 기본 200 QPS

수백만 row 점수화는 **`mlflow.pyfunc.spark_udf`** 로 클러스터에서 병렬 처리.
Endpoint 호출 루프는 **금지**.

In [ ]:
predict_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=f"models:/{model_wheel}/{latest.version}",
    env_manager="virtualenv",
    result_type="int",
)

scored = (
    spark.table(f"{catalog}.{schema}.customers")
         .withColumn("prediction",
                     predict_udf("age", "tenure_months", "monthly_charges",
                                 "total_charges", "support_tickets"))
)
display(scored.limit(10))

# Delta 로 저장
(
    scored.select("customer_id", "prediction")
          .write.mode("overwrite")
          .saveAsTable(f"{catalog}.{schema}.churn_scores_batch")
)
print("✓ batch scoring → churn_scores_batch")

## Step 8. (선택) Endpoint 삭제

데모 후 비용 절약. `99_cleanup` 에서 일괄 정리 가능.

In [ ]:
# w.serving_endpoints.delete(name=endpoint_wheel)
# print(f"✓ deleted {endpoint_wheel}")

## 정리

| 작업 | 도구 |
| --- | --- |
| Endpoint 생성/업데이트 | `WorkspaceClient.serving_endpoints` |
| 노트북 호출 | `mlflow.deployments.get_deploy_client("databricks")` |
| 외부 호출 | REST POST → `/serving-endpoints/<name>/invocations` |
| 자동 로깅 | AI Gateway Inference Tables |
| Rate limit | `AiGatewayConfig.rate_limits` (per-endpoint / per-user) |
| Blue/Green | `update_config_and_wait` + `TrafficConfig` |
| Batch | `mlflow.pyfunc.spark_udf` — **endpoint 루프 절대 X** |

→ 다음: **`07_express_deployment`** — Serverless 노트북 + env_pack 으로 원클릭 서빙